In this [article](https://machinelearningmastery.com/build-semantic-search-with-llm-embeddings/), you will learn how to build a simple semantic search engine using sentence embeddings and nearest neighbors.

Topics we will cover include:

*   Generating text embeddings with a sentence transformer model.
*   Implementing a nearest-neighbor semantic search pipeline in Python.


Let’s get started.
![image](https://machinelearningmastery.com/wp-content/uploads/2026/03/mlm-building-simple-semantic-search-engine-hero.png)


# Let's dive step by step

In [1]:

import pandas as pd
import json
from pydantic import BaseModel, Field
from openai import OpenAI
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

We will use a toy public dataset called "ag_news", which contains texts from news articles. The following code loads the dataset and selects the first 1000 articles.

In [2]:

from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors

We now load the dataset and extract the "text" column, which contains the article content. Afterwards, we print a short sample from the first article to inspect the data:

In [3]:

print("Loading dataset...")
dataset = load_dataset("ag_news", split="train[:1000]")

# Extract the text column into a Python list
documents = dataset["text"]

print(f"Loaded {len(documents)} documents.")
print(f"Sample: {documents[0][:100]}...")

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Loaded 1000 documents.
Sample: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\b...


The next step is to obtain embedding vectors (numerical representations) for our 1000 texts. As mentioned earlier, some LLMs are trained specifically to translate text into numerical vectors that capture semantic characteristics. Hugging Face sentence transformer models, such as "all-MiniLM-L6-v2", are a common choice. The following code initializes the model and encodes the batch of text documents into embeddings.

In [4]:
print("Loading embedding model...")
model = SentenceTransformer("all-MiniLM-L6-v2")

# Convert text documents into numerical vector embeddings
print("Encoding documents (this may take a few seconds)...")
document_embeddings = model.encode(documents, show_progress_bar=True)

print(f"Created {document_embeddings.shape[0]} embeddings.")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding documents (this may take a few seconds)...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Created 1000 embeddings.


Next, we initialize a NearestNeighbors object, which implements a nearest-neighbor strategy to find the k most similar documents to a given query. In terms of embeddings, this means identifying the closest vectors (smallest angular distance). We use the cosine metric, where more similar vectors have smaller cosine distances (and higher cosine similarity values).



In [5]:

search_engine = NearestNeighbors(n_neighbors=5, metric="cosine")

search_engine.fit(document_embeddings)
print("Search engine is ready!")

Search engine is ready!


The core logic of our search engine is encapsulated in the following function. It takes a plain-text query, specifies how many top results to retrieve via top_k, computes the query embedding, and retrieves the nearest neighbors from the index.

The loop inside the function prints the top-k results ranked by similarity:

In [6]:
def semantic_search(query, top_k=3):
    # Embed the incoming search query
    query_embedding = model.encode([query])

    # Retrieve the closest matches
    distances, indices = search_engine.kneighbors(query_embedding, n_neighbors=top_k)

    print(f"\n🔍 Query: '{query}'")
    print("-" * 50)

    for i in range(top_k):
        doc_idx = indices[0][i]
        # Convert cosine distance to similarity (1 - distance)
        similarity = 1 - distances[0][i]

        print(f"Result {i+1} (Similarity: {similarity:.4f})")
        print(f"Text: {documents[int(doc_idx)][:150]}...\n")

And that’s it. To test the function, we can formulate a couple of example search queries:

In [7]:
semantic_search("Wall street and stock market trends")
semantic_search("Space exploration and rocket launches")


🔍 Query: 'Wall street and stock market trends'
--------------------------------------------------
Result 1 (Similarity: 0.6258)
Text: Stocks Higher Despite Soaring Oil Prices NEW YORK - Wall Street shifted higher Monday as bargain hunters shrugged off skyrocketing oil prices and boug...

Result 2 (Similarity: 0.5586)
Text: Stocks Sharply Higher on Dip in Oil Prices NEW YORK - A drop in oil prices and upbeat outlooks from Wal-Mart and Lowe's prompted new bargain-hunting o...

Result 3 (Similarity: 0.5459)
Text: Strategies for a Sideways Market (Reuters) Reuters - The bulls and the bears are in this\together, scratching their heads and wondering what's going t...


🔍 Query: 'Space exploration and rocket launches'
--------------------------------------------------
Result 1 (Similarity: 0.5803)
Text: Redesigning Rockets: NASA Space Propulsion Finds a New Home (SPACE.com) SPACE.com - While the exploration of the Moon and other planets in our solar s...

Result 2 (Similarity: 0.5008)
Text: 

# Summary
What we have built here can be seen as a gateway to retrieval augmented generation systems. While this example is intentionally simple, semantic search engines like this form the foundational retrieval layer in modern architectures that combine semantic search with large language models.

Now that you know how to build a basic semantic search engine, you may want to explore retrieval augmented generation systems in more depth.